In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split
import json
import logging
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import os
from scipy import stats
from sentence_transformers import CrossEncoder
import warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class RerankingExample:
    """Training example for reranking model"""
    query: str
    document_text: str
    doc_id: str
    relevance_score: float  # 0-1 relevance score
    metadata_features: Dict[str, Any] = field(default_factory=dict)

@dataclass
class RerankingBatch:
    """Batch of documents for a single query"""
    query: str
    documents: List[RerankingExample]
    query_id: str = ""

class LegalRerankingDataset(Dataset):
    """PyTorch dataset for legal document reranking"""
    
    def __init__(self, 
                 batches: List[RerankingBatch], 
                 tokenizer, 
                 max_length: int = 512,
                 include_metadata: bool = True):
        self.batches = batches
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.include_metadata = include_metadata
        
        # Flatten batches into individual examples
        self.examples = []
        for batch in batches:
            for doc in batch.documents:
                self.examples.append((batch.query, doc))
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        query, document = self.examples[idx]
        
        # Tokenize the pair
        encoding = self.tokenizer(
            query,
            document.document_text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Extract metadata features with improved normalization
        metadata_features = torch.zeros(12)  # Expanded feature vector
        if self.include_metadata and document.metadata_features:
            # Court level mapping with more granular levels
            court_mapping = {
                'Supreme Court': 1.0,
                'Court of Appeal': 0.8,
                'High Court': 0.6,
                'District Court': 0.4,
                'Magistrate\'s Court': 0.2,
                'Primary Court': 0.1
            }
            jurisdiction = document.metadata_features.get('jurisdiction', 'District Court')
            metadata_features[0] = court_mapping.get(jurisdiction, 0.4)
            
            # Citation count (log-normalized)
            citation_count = document.metadata_features.get('citation_count', 0)
            metadata_features[1] = min(np.log1p(citation_count) / np.log1p(100), 1.0)
            
            # Authority score
            metadata_features[2] = document.metadata_features.get('authority_score', 0.0)
            
            # Recency boost (exponential decay)
            recency = document.metadata_features.get('recency_boost', 0.0)
            metadata_features[3] = min(recency, 1.0)
            
            # Document type encoding
            doc_type = document.metadata_features.get('doc_type', 'case')
            if doc_type == 'act':
                metadata_features[4] = 1.0
            elif doc_type == 'regulation':
                metadata_features[4] = 0.8
            elif doc_type == 'case':
                metadata_features[4] = 0.6
            else:
                metadata_features[4] = 0.3
            
            # Legal domain match count
            domains = document.metadata_features.get('legal_domains', [])
            metadata_features[5] = min(len(domains) / 5.0, 1.0)
            
            # Jurisdiction match
            metadata_features[6] = 1.0 if document.metadata_features.get('jurisdiction_match', False) else 0.0
            
            # Act references count
            act_refs = document.metadata_features.get('act_references', 0)
            metadata_features[7] = min(act_refs / 10.0, 1.0)
            
            # Judge count
            judge_count = document.metadata_features.get('judge_count', 1)
            metadata_features[8] = min(judge_count / 5.0, 1.0)
            
            # Vector similarity score
            metadata_features[9] = document.metadata_features.get('vector_score', 0.0)
            
            # Professional conduct indicator
            domains_str = ' '.join(domains).lower()
            metadata_features[10] = 1.0 if 'professional conduct' in domains_str else 0.0
            
            # Date recency (based on year)
            date_str = document.metadata_features.get('date', '2020-01-01')
            try:
                year = int(date_str.split('-')[0])
                # Normalize years 2020-2024 to 0-1 scale
                metadata_features[11] = max(0.0, min(1.0, (year - 2020) / 4.0))
            except:
                metadata_features[11] = 0.5
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'token_type_ids': encoding.get('token_type_ids', torch.zeros_like(encoding['input_ids'])).squeeze(),
            'metadata_features': metadata_features,
            'relevance_score': torch.tensor(document.relevance_score, dtype=torch.float),
            'doc_id': document.doc_id,
            'query': query,
            'document_text': document.document_text
        }

class ImprovedNeuralLegalReranker(nn.Module):
    """Improved Neural Legal Reranker with enhanced architecture"""
    
    def __init__(self, 
                 model_name: str = "nlpaueb/legal-bert-base-uncased",
                 metadata_dim: int = 12,
                 hidden_dim: int = 256,
                 dropout_rate: float = 0.15,
                 combine_strategy: str = "attention_fusion"):
        super().__init__()
        
        self.model_name = model_name
        self.combine_strategy = combine_strategy
        
        # Initialize tokenizer with fallback
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.transformer = AutoModel.from_pretrained(model_name)
            logger.info(f"Loaded model: {model_name}")
        except Exception as e:
            logger.warning(f"Could not load {model_name}, falling back to BERT base: {e}")
            self.tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
            self.transformer = AutoModel.from_pretrained("bert-base-uncased")
            self.model_name = "bert-base-uncased"
        
        # Enhanced semantic processing
        self.semantic_processor = nn.Sequential(
            nn.Linear(self.transformer.config.hidden_size, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Enhanced metadata processing with attention
        self.metadata_processor = nn.Sequential(
            nn.Linear(metadata_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate // 2),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Attention-based fusion mechanism
        if combine_strategy == "attention_fusion":
            self.attention_weights = nn.Sequential(
                nn.Linear(2, hidden_dim // 4),
                nn.Tanh(),
                nn.Linear(hidden_dim // 4, 2),
                nn.Softmax(dim=1)
            )
        elif combine_strategy == "cross_attention":
            self.cross_attention = nn.MultiheadAttention(
                embed_dim=hidden_dim // 2, 
                num_heads=4, 
                dropout=dropout_rate,
                batch_first=True
            )
            self.final_projection = nn.Linear(hidden_dim // 2, 1)
        else:
            # Learnable weights for weighted combination
            self.semantic_weight = nn.Parameter(torch.tensor(0.7))
            self.metadata_weight = nn.Parameter(torch.tensor(0.3))
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights with Xavier initialization"""
        for module in [self.semantic_processor, self.metadata_processor]:
            for m in module:
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    nn.init.zeros_(m.bias)
    
    def forward(self, input_ids, attention_mask, token_type_ids, metadata_features):
        batch_size = input_ids.size(0)
        
        # Get transformer outputs with attention pooling
        transformer_outputs = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        
        # Enhanced semantic representation using weighted average of all tokens
        hidden_states = transformer_outputs.last_hidden_state
        attention_weights = attention_mask.unsqueeze(-1).float()
        weighted_embeddings = (hidden_states * attention_weights).sum(dim=1) / attention_weights.sum(dim=1).clamp(min=1e-8)
        
        # Get semantic score
        semantic_score = self.semantic_processor(weighted_embeddings)
        
        # Get metadata score
        metadata_score = self.metadata_processor(metadata_features)
        
        # Combine scores based on strategy
        if self.combine_strategy == "attention_fusion":
            # Learn attention weights over semantic and metadata scores
            score_concat = torch.cat([semantic_score, metadata_score], dim=1)
            attention_weights = self.attention_weights(score_concat)
            final_score = (attention_weights * score_concat).sum(dim=1, keepdim=True)
            
        elif self.combine_strategy == "cross_attention":
            # Cross-attention between semantic and metadata representations
            semantic_repr = weighted_embeddings.unsqueeze(1)  # [batch, 1, hidden]
            metadata_repr = self.metadata_processor[:-1](metadata_features).unsqueeze(1)  # [batch, 1, hidden//2]
            
            # Pad metadata to match semantic dimension
            pad_size = semantic_repr.size(-1) - metadata_repr.size(-1)
            metadata_repr = torch.cat([metadata_repr, torch.zeros(batch_size, 1, pad_size, device=metadata_repr.device)], dim=-1)
            
            attended_repr, _ = self.cross_attention(semantic_repr, metadata_repr, metadata_repr)
            final_score = self.final_projection(attended_repr.squeeze(1))
            
        else:
            # Weighted combination with learnable parameters
            weights = torch.softmax(torch.stack([self.semantic_weight, self.metadata_weight]), dim=0)
            final_score = weights[0] * semantic_score + weights[1] * metadata_score
        
        return torch.sigmoid(final_score)  # Ensure output is in [0,1] range
    
    def predict_scores(self, query_doc_pairs: List[Tuple[str, str]], metadata_list: List[Dict] = None):
        """Predict relevance scores for query-document pairs"""
        self.eval()
        
        if metadata_list is None:
            metadata_list = [{}] * len(query_doc_pairs)
        
        # Process in batches for efficiency
        batch_size = 16
        all_scores = []
        
        for i in range(0, len(query_doc_pairs), batch_size):
            batch_pairs = query_doc_pairs[i:i+batch_size]
            batch_metadata = metadata_list[i:i+batch_size]
            
            # Tokenize batch
            inputs = self.tokenizer(
                [pair[0] for pair in batch_pairs],
                [pair[1] for pair in batch_pairs],
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors='pt'
            )
            
            # Process metadata batch
            metadata_tensor = torch.zeros(len(batch_pairs), 12)
            for j, metadata in enumerate(batch_metadata):
                if metadata:
                    # Apply same processing as in dataset
                    court_mapping = {
                        'Supreme Court': 1.0, 'Court of Appeal': 0.8, 'High Court': 0.6,
                        'District Court': 0.4, 'Magistrate\'s Court': 0.2, 'Primary Court': 0.1
                    }
                    jurisdiction = metadata.get('jurisdiction', 'District Court')
                    metadata_tensor[j, 0] = court_mapping.get(jurisdiction, 0.4)
                    metadata_tensor[j, 1] = min(np.log1p(metadata.get('citation_count', 0)) / np.log1p(100), 1.0)
                    metadata_tensor[j, 2] = metadata.get('authority_score', 0.0)
                    metadata_tensor[j, 3] = min(metadata.get('recency_boost', 0.0), 1.0)
                    
                    doc_type = metadata.get('doc_type', 'case')
                    type_mapping = {'act': 1.0, 'regulation': 0.8, 'case': 0.6}
                    metadata_tensor[j, 4] = type_mapping.get(doc_type, 0.3)
                    
                    domains = metadata.get('legal_domains', [])
                    metadata_tensor[j, 5] = min(len(domains) / 5.0, 1.0)
                    metadata_tensor[j, 6] = 1.0 if metadata.get('jurisdiction_match', False) else 0.0
                    metadata_tensor[j, 7] = min(metadata.get('act_references', 0) / 10.0, 1.0)
                    metadata_tensor[j, 8] = min(metadata.get('judge_count', 1) / 5.0, 1.0)
                    metadata_tensor[j, 9] = metadata.get('vector_score', 0.0)
                    
                    domains_str = ' '.join(domains).lower()
                    metadata_tensor[j, 10] = 1.0 if 'professional conduct' in domains_str else 0.0
                    
                    try:
                        year = int(metadata.get('date', '2020-01-01').split('-')[0])
                        metadata_tensor[j, 11] = max(0.0, min(1.0, (year - 2020) / 4.0))
                    except:
                        metadata_tensor[j, 11] = 0.5
            
            with torch.no_grad():
                device = next(self.parameters()).device
                input_ids = inputs['input_ids'].to(device)
                attention_mask = inputs['attention_mask'].to(device)
                token_type_ids = inputs.get('token_type_ids', torch.zeros_like(input_ids)).to(device)
                metadata_tensor = metadata_tensor.to(device)
                
                batch_scores = self.forward(input_ids, attention_mask, token_type_ids, metadata_tensor)
                all_scores.extend(batch_scores.cpu().numpy().flatten())
        
        return np.array(all_scores)

class ImprovedRerankingTrainer:
    """Enhanced trainer with advanced optimization and evaluation"""
    
    def __init__(self, 
                 model: ImprovedNeuralLegalReranker,
                 train_loader: DataLoader,
                 val_loader: DataLoader,
                 learning_rate: float = 2e-5,
                 weight_decay: float = 1e-5,
                 device: str = None,
                 use_focal_loss: bool = True,
                 focal_alpha: float = 0.25,
                 focal_gamma: float = 2.0):
        
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = model.to(self.device)
        self.use_focal_loss = use_focal_loss
        self.focal_alpha = focal_alpha
        self.focal_gamma = focal_gamma
        
        self.train_loader = train_loader
        self.val_loader = val_loader
        
        # Advanced optimizer with different learning rates
        transformer_params = []
        other_params = []

        for name, param in self.model.named_parameters():
            if 'transformer' in name:
                transformer_params.append(param)
            else:
                other_params.append(param)
        
        self.optimizer = optim.AdamW([
            {'params': transformer_params, 'lr': learning_rate * 0.1},  # Lower LR for pretrained
            {'params': other_params, 'lr': learning_rate},
        ], weight_decay=weight_decay)
        
        # Loss functions
        self.mse_loss = nn.MSELoss()
        self.bce_loss = nn.BCELoss()
        self.margin_ranking_loss = nn.MarginRankingLoss(margin=0.1)
        
        # Training history
        self.training_history = {
            'train_loss': [], 'val_loss': [], 'ndcg_at_5': [], 'ndcg_at_10': [],
            'map_score': [], 'spearman_corr': [], 'mrr': [], 'precision_at_5': []
        }
        
        # Advanced scheduler
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, T_0=10, T_mult=2, eta_min=1e-7
        )
    
    def focal_loss(self, predictions, targets):
        """Focal loss for handling class imbalance"""
        bce = self.bce_loss(predictions, targets)
        p_t = torch.where(targets == 1, predictions, 1 - predictions)
        alpha_t = torch.where(targets == 1, self.focal_alpha, 1 - self.focal_alpha)
        focal_weight = alpha_t * (1 - p_t) ** self.focal_gamma
        return focal_weight * bce
    
    def listwise_loss(self, predictions, relevance_scores, temperature=1.0):
        """ListNet loss for listwise learning"""
        # Apply temperature scaling
        predictions = predictions / temperature
        relevance_scores = relevance_scores / temperature
        
        # Compute softmax probabilities
        pred_probs = torch.softmax(predictions, dim=0)
        true_probs = torch.softmax(relevance_scores, dim=0)
        
        # KL divergence loss
        return torch.sum(true_probs * torch.log(true_probs / (pred_probs + 1e-8) + 1e-8))
    
    def train_epoch(self):
        """Enhanced training epoch with multiple loss components"""
        self.model.train()
        total_loss = 0.0
        num_batches = 0
        
        for batch in self.train_loader:
            # Move to device
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            token_type_ids = batch['token_type_ids'].to(self.device)
            metadata_features = batch['metadata_features'].to(self.device)
            relevance_scores = batch['relevance_score'].to(self.device)
            
            # Forward pass
            predictions = self.model(input_ids, attention_mask, token_type_ids, metadata_features).squeeze()
            
            # Combined loss
            if self.use_focal_loss:
                # Convert to binary classification for focal loss
                binary_targets = (relevance_scores > 0.5).float()
                main_loss = self.focal_loss(predictions, binary_targets)
            else:
                main_loss = self.mse_loss(predictions, relevance_scores)
            
            loss = main_loss
            
            # Add listwise loss if batch size > 1
            if len(predictions) > 1:
                listwise_loss = self.listwise_loss(predictions, relevance_scores)
                loss += 0.2 * listwise_loss
            
            # Add pairwise ranking loss
            if len(predictions) > 1:
                ranking_loss = self.pairwise_ranking_loss(predictions, relevance_scores)
                loss += 0.3 * ranking_loss
            
            # Backward pass with gradient clipping
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
        
        self.scheduler.step()
        return total_loss / num_batches
    
    def pairwise_ranking_loss(self, predictions, relevance_scores, margin=0.1):
        """Enhanced pairwise ranking loss"""
        batch_size = predictions.size(0)
        if batch_size < 2:
            return torch.tensor(0.0, device=predictions.device)
        
        loss = 0.0
        num_pairs = 0
        
        for i in range(batch_size):
            for j in range(i + 1, batch_size):
                rel_diff = relevance_scores[i] - relevance_scores[j]
                
                # Only consider pairs with significant relevance difference
                if abs(rel_diff) > 0.05:
                    pred_diff = predictions[i] - predictions[j]
                    
                    if rel_diff > 0:  # i should be ranked higher than j
                        loss += torch.clamp(margin - pred_diff, min=0.0)
                    else:  # j should be ranked higher than i
                        loss += torch.clamp(margin + pred_diff, min=0.0)
                    
                    num_pairs += 1
        
        return loss / max(num_pairs, 1)
    
    def evaluate(self, loader: DataLoader, detailed=False):
        """Comprehensive evaluation with multiple metrics"""
        self.model.eval()
        total_loss = 0.0
        all_predictions = []
        all_relevance = []
        all_queries = []
        all_doc_ids = []
        
        with torch.no_grad():
            for batch in loader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                token_type_ids = batch['token_type_ids'].to(self.device)
                metadata_features = batch['metadata_features'].to(self.device)
                relevance_scores = batch['relevance_score'].to(self.device)
                
                predictions = self.model(input_ids, attention_mask, token_type_ids, metadata_features).squeeze()
                
                # Calculate loss (same as training)
                if self.use_focal_loss:
                    binary_targets = (relevance_scores > 0.5).float()
                    main_loss = self.focal_loss(predictions, binary_targets)
                else:
                    main_loss = self.mse_loss(predictions, relevance_scores)
                
                loss = main_loss
                if len(predictions) > 1:
                    listwise_loss = self.listwise_loss(predictions, relevance_scores)
                    ranking_loss = self.pairwise_ranking_loss(predictions, relevance_scores)
                    loss += 0.2 * listwise_loss + 0.3 * ranking_loss
                
                total_loss += loss.item()
                
                # Store for metrics calculation
                all_predictions.extend(predictions.cpu().numpy().flatten())
                all_relevance.extend(relevance_scores.cpu().numpy().flatten())
                all_queries.extend(batch['query'])
                all_doc_ids.extend(batch['doc_id'])
        
        avg_loss = total_loss / len(loader)
        metrics = self.calculate_comprehensive_metrics(all_predictions, all_relevance, all_queries, all_doc_ids)
        
        if detailed:
            self.create_evaluation_report(all_predictions, all_relevance, all_queries, all_doc_ids)
        
        return avg_loss, metrics
    
    def calculate_comprehensive_metrics(self, predictions, relevance_scores, queries, doc_ids):
        """Calculate comprehensive ranking metrics"""
        # Group by query
        query_groups = defaultdict(list)
        for pred, rel, query, doc_id in zip(predictions, relevance_scores, queries, doc_ids):
            query_groups[query].append({
                'prediction': pred, 'relevance': rel, 'doc_id': doc_id
            })
        
        metrics = {
            'ndcg_at_5': [], 'ndcg_at_10': [], 'map_score': [], 'spearman_corr': [],
            'mrr': [], 'precision_at_5': [], 'recall_at_10': [], 'f1_at_5': []
        }
        
        for query, docs in query_groups.items():
            if len(docs) < 2:
                continue
            
            # Sort by predicted relevance (descending)
            docs_sorted = sorted(docs, key=lambda x: x['prediction'], reverse=True)
            
            true_relevance = [doc['relevance'] for doc in docs_sorted]
            predicted_scores = [doc['prediction'] for doc in docs_sorted]
            
            # NDCG scores
            if len(true_relevance) >= 5:
                ndcg_5 = ndcg_score([true_relevance], [predicted_scores], k=5)
                metrics['ndcg_at_5'].append(ndcg_5)
            
            if len(true_relevance) >= 10:
                ndcg_10 = ndcg_score([true_relevance], [predicted_scores], k=10)
                metrics['ndcg_at_10'].append(ndcg_10)
            
            # MAP
            ap_score = self.average_precision(true_relevance)
            metrics['map_score'].append(ap_score)
            
            # MRR (Mean Reciprocal Rank)
            mrr = self.reciprocal_rank(true_relevance)
            metrics['mrr'].append(mrr)
            
            # Precision@5
            precision_5 = self.precision_at_k(true_relevance, k=5)
            metrics['precision_at_5'].append(precision_5)
            
            # Recall@10
            recall_10 = self.recall_at_k(true_relevance, k=10)
            metrics['recall_at_10'].append(recall_10)
            
            # F1@5
            f1_5 = self.f1_at_k(true_relevance, k=5)
            metrics['f1_at_5'].append(f1_5)
            
            # Spearman correlation
            original_relevance = [doc['relevance'] for doc in docs]
            original_predictions = [doc['prediction'] for doc in docs]
            if len(set(original_relevance)) > 1:
                corr, _ = stats.spearmanr(original_predictions, original_relevance)
                if not np.isnan(corr):
                    metrics['spearman_corr'].append(corr)
        
        # Average all metrics
        return {key: np.mean(values) if values else 0.0 for key, values in metrics.items()}
    
    def average_precision(self, relevance_scores, threshold=0.5):
        """Calculate Average Precision"""
        relevant_count = 0
        precision_sum = 0.0
        
        for i, score in enumerate(relevance_scores):
            if score > threshold:
                relevant_count += 1
                precision_at_i = relevant_count / (i + 1)
                precision_sum += precision_at_i
        
        total_relevant = sum(1 for score in relevance_scores if score > threshold)
        return precision_sum / total_relevant if total_relevant > 0 else 0.0
    
    def reciprocal_rank(self, relevance_scores, threshold=0.5):
        """Calculate Reciprocal Rank"""
        for i, score in enumerate(relevance_scores):
            if score > threshold:
                return 1.0 / (i + 1)
        return 0.0
    
    def precision_at_k(self, relevance_scores, k=5, threshold=0.5):
        """Calculate Precision@K"""
        top_k = relevance_scores[:min(k, len(relevance_scores))]
        relevant_count = sum(1 for score in top_k if score > threshold)
        return relevant_count / len(top_k) if top_k else 0.0
    
    def recall_at_k(self, relevance_scores, k=10, threshold=0.5):
        """Calculate Recall@K"""
        top_k = relevance_scores[:min(k, len(relevance_scores))]
        relevant_in_top_k = sum(1 for score in top_k if score > threshold)
        total_relevant = sum(1 for score in relevance_scores if score > threshold)
        return relevant_in_top_k / total_relevant if total_relevant > 0 else 0.0
    
    def f1_at_k(self, relevance_scores, k=5, threshold=0.5):
        """Calculate F1@K"""
        precision = self.precision_at_k(relevance_scores, k, threshold)
        recall = self.recall_at_k(relevance_scores, k, threshold)
        return 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    def create_evaluation_report(self, predictions, relevance_scores, queries, doc_ids):
        """Create detailed evaluation report with visualizations"""
        print("\n" + "="*80)
        print("DETAILED EVALUATION REPORT")
        print("="*80)
        
        # Create DataFrame for analysis
        df = pd.DataFrame({
            'prediction': predictions,
            'relevance': relevance_scores,
            'query': queries,
            'doc_id': doc_ids
        })
        
        # Overall statistics
        print(f"\nOVERALL STATISTICS:")
        print(f"Total documents evaluated: {len(df)}")
        print(f"Unique queries: {len(df['query'].unique())}")
        print(f"Average documents per query: {len(df) / len(df['query'].unique()):.2f}")
        print(f"Prediction range: [{df['prediction'].min():.3f}, {df['prediction'].max():.3f}]")
        print(f"Relevance range: [{df['relevance'].min():.3f}, {df['relevance'].max():.3f}]")
        print(f"Overall correlation: {df['prediction'].corr(df['relevance']):.4f}")
        
        # Query-level analysis
        query_stats = []
        for query in df['query'].unique():
            query_df = df[df['query'] == query]
            corr = query_df['prediction'].corr(query_df['relevance'])
            query_stats.append({
                'query': query[:50] + "..." if len(query) > 50 else query,
                'num_docs': len(query_df),
                'correlation': corr if not np.isnan(corr) else 0.0,
                'avg_relevance': query_df['relevance'].mean(),
                'avg_prediction': query_df['prediction'].mean()
            })
        
        query_stats_df = pd.DataFrame(query_stats)
        print(f"\nQUERY-LEVEL ANALYSIS:")
        print(f"Best performing queries (by correlation):")
        print(query_stats_df.nlargest(3, 'correlation')[['query', 'correlation', 'num_docs']])
        print(f"\nWorst performing queries (by correlation):")
        print(query_stats_df.nsmallest(3, 'correlation')[['query', 'correlation', 'num_docs']])
        
        # Prediction distribution analysis
        print(f"\nPREDICTION DISTRIBUTION:")
        print(f"High relevance (>0.7): {(df['relevance'] > 0.7).sum()} docs")
        print(f"Medium relevance (0.3-0.7): {((df['relevance'] >= 0.3) & (df['relevance'] <= 0.7)).sum()} docs")
        print(f"Low relevance (<0.3): {(df['relevance'] < 0.3).sum()} docs")
        
        # Create visualizations if matplotlib is available
        try:
            plt.figure(figsize=(15, 10))
            
            # Scatter plot of predictions vs relevance
            plt.subplot(2, 3, 1)
            plt.scatter(df['relevance'], df['prediction'], alpha=0.6)
            plt.plot([0, 1], [0, 1], 'r--', label='Perfect correlation')
            plt.xlabel('True Relevance')
            plt.ylabel('Predicted Score')
            plt.title('Predictions vs True Relevance')
            plt.legend()
            
            # Prediction distribution
            plt.subplot(2, 3, 2)
            plt.hist(df['prediction'], bins=20, alpha=0.7, label='Predictions')
            plt.hist(df['relevance'], bins=20, alpha=0.7, label='True Relevance')
            plt.xlabel('Score')
            plt.ylabel('Frequency')
            plt.title('Score Distributions')
            plt.legend()
            
            # Correlation by query
            plt.subplot(2, 3, 3)
            plt.hist(query_stats_df['correlation'], bins=15, alpha=0.7)
            plt.xlabel('Correlation')
            plt.ylabel('Number of Queries')
            plt.title('Query-level Correlation Distribution')
            
            # Error analysis
            plt.subplot(2, 3, 4)
            errors = df['prediction'] - df['relevance']
            plt.hist(errors, bins=20, alpha=0.7)
            plt.xlabel('Error (Pred - True)')
            plt.ylabel('Frequency')
            plt.title('Prediction Error Distribution')
            plt.axvline(x=0, color='r', linestyle='--')
            
            # Query performance
            plt.subplot(2, 3, 5)
            plt.scatter(query_stats_df['num_docs'], query_stats_df['correlation'])
            plt.xlabel('Number of Documents')
            plt.ylabel('Correlation')
            plt.title('Performance vs Query Size')
            
            # Relevance vs prediction by bins
            plt.subplot(2, 3, 6)
            bins = pd.cut(df['relevance'], bins=5)
            bin_stats = df.groupby(bins)['prediction'].mean()
            bin_stats.plot(kind='bar', rot=45)
            plt.xlabel('Relevance Bins')
            plt.ylabel('Average Prediction')
            plt.title('Prediction by Relevance Bins')
            
            plt.tight_layout()
            plt.savefig('evaluation_report.png', dpi=300, bbox_inches='tight')
            plt.show()
            
        except Exception as e:
            print(f"Could not create visualizations: {e}")
    
    def train(self, num_epochs: int, save_path: str = None, early_stopping_patience: int = 7):
        """Train with comprehensive monitoring and early stopping"""
        best_metric = 0.0  # Use NDCG@10 as primary metric
        patience_counter = 0
        
        logger.info(f"Starting training for {num_epochs} epochs on {self.device}")
        logger.info(f"Model: {self.model.model_name}")
        logger.info(f"Training samples: {len(self.train_loader.dataset)}")
        logger.info(f"Validation samples: {len(self.val_loader.dataset)}")
        logger.info(f"Combine strategy: {self.model.combine_strategy}")
        
        for epoch in range(num_epochs):
            # Training
            train_loss = self.train_epoch()
            
            # Validation
            val_loss, val_metrics = self.evaluate(self.val_loader)
            
            # Store metrics
            self.training_history['train_loss'].append(train_loss)
            self.training_history['val_loss'].append(val_loss)
            for key, value in val_metrics.items():
                if key in self.training_history:
                    self.training_history[key].append(value)
            
            # Logging
            logger.info(f"\nEpoch {epoch+1}/{num_epochs}:")
            logger.info(f"  Train Loss: {train_loss:.4f}")
            logger.info(f"  Val Loss: {val_loss:.4f}")
            logger.info(f"  NDCG@5: {val_metrics['ndcg_at_5']:.4f}")
            logger.info(f"  NDCG@10: {val_metrics['ndcg_at_10']:.4f}")
            logger.info(f"  MAP: {val_metrics['map_score']:.4f}")
            logger.info(f"  MRR: {val_metrics['mrr']:.4f}")
            logger.info(f"  Precision@5: {val_metrics['precision_at_5']:.4f}")
            logger.info(f"  Spearman: {val_metrics['spearman_corr']:.4f}")
            
            # Early stopping based on NDCG@10
            current_metric = val_metrics['ndcg_at_10']
            if current_metric > best_metric:
                best_metric = current_metric
                patience_counter = 0
                
                # Save best model
                if save_path:
                    self.save_model(save_path)
                    logger.info(f"Saved best model (NDCG@10: {best_metric:.4f}) to {save_path}")
            else:
                patience_counter += 1
                
            if patience_counter >= early_stopping_patience:
                logger.info(f"Early stopping at epoch {epoch+1} (best NDCG@10: {best_metric:.4f})")
                break
        
        return self.training_history
    
    def save_model(self, path: str):
        """Save model with comprehensive metadata"""
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'model_config': {
                'model_name': self.model.model_name,
                'combine_strategy': self.model.combine_strategy,
                'metadata_dim': 12,
                'hidden_dim': 256
            },
            'optimizer_state_dict': self.optimizer.state_dict(),
            'training_history': self.training_history,
            'scheduler_state_dict': self.scheduler.state_dict()
        }, path)
    
    def load_model(self, path: str):
        """Load model with all components"""
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.training_history = checkpoint.get('training_history', self.training_history)
        if 'scheduler_state_dict' in checkpoint:
            self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

class TrainingDataGenerator:
    """Enhanced training data generator with realistic legal scenarios"""
    
    def __init__(self):
        self.legal_queries = [
            "contempt of court proceedings constitutional provisions",
            "fundamental rights protection judicial review",
            "criminal procedure arrest detention rights",
            "contract law breach damages remedies",
            "tort liability negligence compensation",
            "property rights title succession disputes",
            "administrative law government decisions review",
            "evidence law digital documents admissibility",
            "family law custody maintenance provisions",
            "commercial law company incorporation requirements",
            "Bar Association complaint judicial misconduct allegations",
            "professional conduct legal practitioners discipline",
            "constitutional interpretation fundamental rights violations",
            "criminal appeals conviction sentence review",
            "civil litigation procedural compliance requirements"
        ]
        
        self.legal_domains = [
            'constitutional', 'criminal', 'civil', 'commercial', 'administrative',
            'professional conduct', 'family law', 'property law', 'tort law'
        ]
        
        self.jurisdictions = [
            'Supreme Court', 'Court of Appeal', 'High Court', 
            'District Court', 'Magistrate\'s Court', 'Primary Court'
        ]
    
    def load_from_json(self, json_file_path: str) -> List[RerankingBatch]:
        """Load training data from JSON file"""
        try:
            with open(json_file_path, 'r', encoding='utf-8') as f:
                json_data = json.load(f)
            
            logger.info(f"Loaded {len(json_data)} queries from {json_file_path}")
            
            batches = []
            for item in json_data:
                query = item['query']
                query_id = item['query_id']
                
                documents = []
                for doc_data in item['documents']:
                    # Handle potential missing fields
                    metadata_features = doc_data.get('metadata_features', {})
                    
                    # Enhance metadata with computed features
                    enhanced_metadata = self._enhance_metadata(metadata_features, doc_data.get('document_text', ''))
                    
                    example = RerankingExample(
                        query=query,
                        document_text=doc_data['document_text'],
                        doc_id=doc_data['doc_id'],
                        relevance_score=doc_data['relevance_score'],
                        metadata_features=enhanced_metadata
                    )
                    documents.append(example)
                
                batch = RerankingBatch(query=query, documents=documents, query_id=query_id)
                batches.append(batch)
            
            return batches
            
        except FileNotFoundError:
            logger.error(f"Training data file not found: {json_file_path}")
            logger.info("Generating synthetic training data instead...")
            return self.generate_synthetic_training_data()
        except Exception as e:
            logger.error(f"Error loading training data: {e}")
            logger.info("Generating synthetic training data instead...")
            return self.generate_synthetic_training_data()
    
    def _enhance_metadata(self, metadata: Dict, document_text: str) -> Dict:
        """Enhance metadata with computed features"""
        enhanced = metadata.copy()
        
        # Add default values for missing fields
        enhanced.setdefault('citation_count', 0)
        enhanced.setdefault('authority_score', 0.5)
        enhanced.setdefault('recency_boost', 0.5)
        enhanced.setdefault('doc_type', 'case')
        enhanced.setdefault('jurisdiction_match', False)
        enhanced.setdefault('act_references', 0)
        enhanced.setdefault('judge_count', 1)
        enhanced.setdefault('vector_score', 0.5)
        
        # Compute text-based features
        text_lower = document_text.lower()
        
        # Professional conduct indicator
        conduct_keywords = ['professional conduct', 'misconduct', 'discipline', 'bar association']
        if any(keyword in text_lower for keyword in conduct_keywords):
            if 'legal_domains' not in enhanced:
                enhanced['legal_domains'] = []
            if 'professional conduct' not in enhanced['legal_domains']:
                enhanced['legal_domains'].append('professional conduct')
        
        # Estimate authority based on content
        authority_indicators = ['supreme court', 'constitutional', 'precedent', 'landmark']
        authority_count = sum(1 for indicator in authority_indicators if indicator in text_lower)
        enhanced['authority_score'] = min(1.0, enhanced['authority_score'] + authority_count * 0.1)
        
        return enhanced
    
    def generate_synthetic_training_data(self, num_queries: int = 50, docs_per_query: int = 15) -> List[RerankingBatch]:
        """Generate high-quality synthetic training data"""
        logger.info(f"Generating {num_queries} synthetic queries with {docs_per_query} documents each")
        
        batches = []
        
        for i in range(num_queries):
            query = np.random.choice(self.legal_queries)
            query_id = f"synthetic_query_{i}"
            
            documents = []
            for j in range(docs_per_query):
                # Generate more realistic relevance distribution
                if j < 3:  # Top 3 documents should be highly relevant
                    relevance = np.random.beta(8, 2)  # Skewed towards high relevance
                elif j < 8:  # Next 5 should be moderately relevant
                    relevance = np.random.beta(3, 3)  # More balanced
                else:  # Remaining should be less relevant
                    relevance = np.random.beta(2, 8)  # Skewed towards low relevance
                
                # Generate contextual document text
                query_terms = query.split()
                main_topic = query_terms[0] if query_terms else "legal"
                
                doc_templates = [
                    f"This case discusses {main_topic} in the context of {' '.join(query_terms[1:3])} with detailed analysis of statutory provisions and judicial precedents.",
                    f"Legal analysis of {main_topic} procedures including {' '.join(query_terms[-2:])} as established in constitutional jurisprudence and case law.",
                    f"Court ruling on {main_topic} matters concerning {' '.join(query_terms[1:3])} with extensive citation of relevant legislation and judicial decisions.",
                    f"Judicial interpretation of {main_topic} provisions relating to {' '.join(query_terms[2:4])} in accordance with established legal principles."
                ]
                
                doc_text = np.random.choice(doc_templates)
                
                # Generate realistic metadata
                court_level = np.random.choice([1, 2, 3, 4, 5, 6], p=[0.1, 0.15, 0.25, 0.25, 0.15, 0.1])
                jurisdiction = self.jurisdictions[court_level - 1]
                
                # Higher court levels get more citations and authority
                base_citations = max(0, int(np.random.poisson(20 * (7 - court_level))))
                authority_base = (7 - court_level) / 6.0
                
                metadata_features = {
                    'court_level': court_level,
                    'jurisdiction': jurisdiction,
                    'date': f"202{np.random.randint(0, 5)}-{np.random.randint(1, 13):02d}-{np.random.randint(1, 29):02d}",
                    'legal_domains': np.random.choice(self.legal_domains, size=np.random.randint(1, 4), replace=False).tolist(),
                    'citation_count': base_citations,
                    'authority_score': min(1.0, authority_base + np.random.normal(0, 0.1)),
                    'recency_boost': np.random.exponential(0.3),
                    'doc_type': np.random.choice(['case', 'act', 'regulation'], p=[0.7, 0.2, 0.1]),
                    'domain_match_count': np.random.randint(0, 4),
                    'jurisdiction_match': np.random.choice([True, False], p=[0.6, 0.4]),
                    'act_references': np.random.poisson(2),
                    'judge_count': np.random.choice([1, 3, 5], p=[0.6, 0.3, 0.1]) if court_level <= 3 else 1,
                    'vector_score': min(1.0, max(0.0, relevance + np.random.normal(0, 0.15)))
                }
                
                example = RerankingExample(
                    query=query,
                    document_text=doc_text,
                    doc_id=f"synthetic_doc_{i}_{j}",
                    relevance_score=relevance,
                    metadata_features=metadata_features
                )
                documents.append(example)
            
            batch = RerankingBatch(query=query, documents=documents, query_id=query_id)
            batches.append(batch)
        
        return batches

def create_data_loaders(batches: List[RerankingBatch], 
                       tokenizer, 
                       test_size: float = 0.2, 
                       batch_size: int = 16,
                       max_length: int = 512) -> Tuple[DataLoader, DataLoader]:
    """Create train and validation data loaders with proper splitting"""
    
    # Split at query level to avoid data leakage
    train_batches, val_batches = train_test_split(
        batches, test_size=test_size, random_state=42, shuffle=True
    )
    
    logger.info(f"Split: {len(train_batches)} train queries, {len(val_batches)} val queries")
    
    # Create datasets
    train_dataset = LegalRerankingDataset(train_batches, tokenizer, max_length)
    val_dataset = LegalRerankingDataset(val_batches, tokenizer, max_length)
    
    logger.info(f"Train dataset: {len(train_dataset)} examples")
    logger.info(f"Val dataset: {len(val_dataset)} examples")
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=2,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=2,
        pin_memory=True
    )
    
    return train_loader, val_loader

def main_training_pipeline(training_data_path: str = "training_data.json"):
    """Complete training pipeline with comprehensive evaluation"""
    
    print("🚀 Starting Improved Neural Legal Reranker Training Pipeline")
    print("=" * 80)
    
    # 1. Load training data
    print("\n📊 Loading training data...")
    data_generator = TrainingDataGenerator()
    training_batches = data_generator.load_from_json(training_data_path)
    
    print(f"✅ Loaded {len(training_batches)} query batches")
    total_docs = sum(len(batch.documents) for batch in training_batches)
    print(f"📄 Total documents: {total_docs}")
    print(f"📊 Average docs per query: {total_docs / len(training_batches):.2f}")
    
    # 2. Initialize model
    print("\n🧠 Initializing model...")
    model = ImprovedNeuralLegalReranker(
        model_name="nlpaueb/legal-bert-base-uncased",
        combine_strategy="attention_fusion",  # Try different strategies
        dropout_rate=0.15
    )
    
    print(f"✅ Model initialized: {model.model_name}")
    print(f"🔧 Combination strategy: {model.combine_strategy}")
    
    # 3. Create data loaders
    print("\n📦 Creating data loaders...")
    train_loader, val_loader = create_data_loaders(
        training_batches, 
        model.tokenizer, 
        test_size=0.2, 
        batch_size=8,  # Smaller batch size for stability
        max_length=512
    )
    
    # 4. Initialize trainer
    print("\n🏋️ Initializing trainer...")
    trainer = ImprovedRerankingTrainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        learning_rate=2e-5,
        weight_decay=1e-5,
        use_focal_loss=True
    )
    
    print(f"🎯 Device: {trainer.device}")
    print(f"🔥 Using focal loss: {trainer.use_focal_loss}")
    
    # 5. Train model
    print("\n🚂 Starting training...")
    history = trainer.train(
        num_epochs=5,
        save_path="best_legal_reranker.pth",
        early_stopping_patience=5
    )
    
    # 6. Final evaluation
    print("\n📈 Final evaluation on validation set...")
    final_loss, final_metrics = trainer.evaluate(val_loader, detailed=True)
    
    print(f"\n🎯 FINAL RESULTS:")
    print(f"Validation Loss: {final_loss:.4f}")
    for metric, value in final_metrics.items():
        print(f"{metric.upper()}: {value:.4f}")
    
    # 7. Plot training history
    print("\n📊 Creating training plots...")
    plot_training_history(history)
    
    # 8. Test inference
    print("\n🔍 Testing inference...")
    test_inference(model, training_batches[:2])  # Test on first 2 queries
    
    print("\n✅ Training pipeline completed successfully!")
    return model, trainer, history

def plot_training_history(history: Dict):
    """Plot comprehensive training history"""
    try:
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        
        # Loss curves
        axes[0, 0].plot(history['train_loss'], label='Train Loss', color='blue')
        axes[0, 0].plot(history['val_loss'], label='Val Loss', color='red')
        axes[0, 0].set_title('Training and Validation Loss')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True)
        
        # NDCG scores
        axes[0, 1].plot(history['ndcg_at_5'], label='NDCG@5', color='green')
        axes[0, 1].plot(history['ndcg_at_10'], label='NDCG@10', color='orange')
        axes[0, 1].set_title('NDCG Scores')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('NDCG')
        axes[0, 1].legend()
        axes[0, 1].grid(True)
        
        # MAP and MRR
        axes[0, 2].plot(history['map_score'], label='MAP', color='purple')
        axes[0, 2].plot(history['mrr'], label='MRR', color='brown')
        axes[0, 2].set_title('MAP and MRR')
        axes[0, 2].set_xlabel('Epoch')
        axes[0, 2].set_ylabel('Score')
        axes[0, 2].legend()
        axes[0, 2].grid(True)
        
        # Precision metrics
        axes[1, 0].plot(history['precision_at_5'], label='Precision@5', color='red')
        axes[1, 0].set_title('Precision@5')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Precision')
        axes[1, 0].legend()
        axes[1, 0].grid(True)
        
        # Correlation
        axes[1, 1].plot(history['spearman_corr'], label='Spearman Correlation', color='teal')
        axes[1, 1].set_title('Spearman Correlation')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Correlation')
        axes[1, 1].legend()
        axes[1, 1].grid(True)
        
        # Combined metrics
        axes[1, 2].plot(history['ndcg_at_10'], label='NDCG@10', alpha=0.7)
        axes[1, 2].plot(history['map_score'], label='MAP', alpha=0.7)
        axes[1, 2].plot(history['spearman_corr'], label='Spearman', alpha=0.7)
        axes[1, 2].set_title('Key Metrics Overview')
        axes[1, 2].set_xlabel('Epoch')
        axes[1, 2].set_ylabel('Score')
        axes[1, 2].legend()
        axes[1, 2].grid(True)
        
        plt.tight_layout()
        plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
        plt.show()
        
    except Exception as e:
        print(f"Could not create training plots: {e}")

def test_inference(model: ImprovedNeuralLegalReranker, test_batches: List[RerankingBatch]):
    """Test model inference on sample queries"""
    print("\n🔍 INFERENCE TESTING")
    print("-" * 50)
    
    model.eval()
    
    for batch in test_batches:
        print(f"\nQuery: {batch.query}")
        print(f"Query ID: {batch.query_id}")
        print(f"Documents: {len(batch.documents)}")
        
        # Prepare data for inference
        query_doc_pairs = [(batch.query, doc.document_text) for doc in batch.documents]
        metadata_list = [doc.metadata_features for doc in batch.documents]
        
        # Get predictions
        predictions = model.predict_scores(query_doc_pairs, metadata_list)
        
        # Sort by prediction score
        doc_scores = list(zip(batch.documents, predictions))
        doc_scores.sort(key=lambda x: x[1], reverse=True)
        
        print("\nTop 5 ranked documents:")
        for i, (doc, pred_score) in enumerate(doc_scores[:5]):
            print(f"{i+1}. Doc ID: {doc.doc_id}")
            print(f"   Predicted: {pred_score:.4f} | True: {doc.relevance_score:.4f}")
            print(f"   Text: {doc.document_text[:100]}...")
            print(f"   Jurisdiction: {doc.metadata_features.get('jurisdiction', 'N/A')}")
            print()

In [ ]:
# Load and train the model
model, trainer, history = main_training_pipeline("training_data.json")

# Use for inference
query = "contempt of court proceedings constitutional provisions"
documents = [("doc1", "Court discusses contempt proceedings..."), 
             ("doc2", "Constitutional law analysis...")]
metadata = [{}, {}]  # Your document metadata

scores = model.predict_scores(documents, metadata)
print(f"Relevance scores: {scores}")

2025-07-25 10:10:49,235 - INFO - Loaded 19 queries from training_data.json


🚀 Starting Improved Neural Legal Reranker Training Pipeline

📊 Loading training data...
✅ Loaded 19 query batches
📄 Total documents: 19
📊 Average docs per query: 1.00

🧠 Initializing model...


2025-07-25 10:10:59,015 - INFO - Loaded model: nlpaueb/legal-bert-base-uncased
2025-07-25 10:10:59,023 - INFO - Split: 15 train queries, 4 val queries
2025-07-25 10:10:59,023 - INFO - Train dataset: 15 examples
2025-07-25 10:10:59,023 - INFO - Val dataset: 4 examples


✅ Model initialized: nlpaueb/legal-bert-base-uncased
🔧 Combination strategy: attention_fusion

📦 Creating data loaders...

🏋️ Initializing trainer...


2025-07-25 10:11:00,606 - INFO - Starting training for 20 epochs on cuda
2025-07-25 10:11:00,606 - INFO - Model: nlpaueb/legal-bert-base-uncased
2025-07-25 10:11:00,606 - INFO - Training samples: 15
2025-07-25 10:11:00,606 - INFO - Validation samples: 4
2025-07-25 10:11:00,609 - INFO - Combine strategy: attention_fusion


🎯 Device: cuda
🔥 Using focal loss: True

🚂 Starting training...


In [ ]:
# Attention-based fusion (recommended)
model = ImprovedNeuralLegalReranker(combine_strategy="attention_fusion")

# Cross-attention fusion (for complex interactions)
model = ImprovedNeuralLegalReranker(combine_strategy="cross_attention")

# Simple weighted combination (baseline)
model = ImprovedNeuralLegalReranker(combine_strategy="weighted_sum")

In [ ]:
# Conservative training (stable but slower)
trainer = ImprovedRerankingTrainer(
    learning_rate=1e-5,
    use_focal_loss=False,
    weight_decay=1e-4
)

# Aggressive training (faster convergence, may overfit)
trainer = ImprovedRerankingTrainer(
    learning_rate=5e-5,
    use_focal_loss=True,
    focal_gamma=3.0
)